In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
from matplotlib import pyplot as plt
jax.config.update("jax_enable_x64", True)

from src import lisa

In [ ]:
TOBS = lisa.YEAR_s
DT = lisa.SAMPLING_STEP_s
NSAMPLES = int(TOBS / DT)
FREQS = jnp.fft.rfftfreq(NSAMPLES, DT)

N_SOURCES = 3

u = jr.uniform(jr.key(0), shape=(N_SOURCES, 8))
params = lisa.prior_inverse_cdf(u)

signal = lisa.clean_signal(params, t_obs=TOBS, dt=DT)
noise_t = lisa.sample_noise(jr.key(1), t_obs=TOBS, dt=DT)
noise_f = jnp.fft.rfft(noise_t)
datastream = signal + noise_f
print(signal.shape, noise_f.shape)

In [ ]:
plt.figure(figsize=(12, 8))
for j, channel in enumerate("AET"):
    plt.subplot(331+j)
    plt.title(f"channel {channel}")
    plt.loglog(FREQS, jnp.abs(datastream[j]), alpha=0.3, label="datastream")
    plt.loglog(FREQS, jnp.abs(signal[j]), alpha=0.3, label="signal")
    plt.xlim(params[:, 0].min(axis=0) * 0.9, params[:, 0].max(axis=0) * 1.1)
    plt.legend()
plt.show()